# Quickstart: Google WeatherNext 3 forecast, 15 day, 0.1 degree, virtual - dynamical.org Icechunk Zarr

This product contains 2026 forecasts and ongoing updates, initialized every six hours, with hourly leads from +1 to +360 hours. Google's precomputed ensemble mean and percentiles are separate `statistic` labels; they are not individual ensemble members. The 0.1° product contains gridded surface weather fields, including temperature, wind, precipitation and radiation.

Dataset documentation: https://dynamical.org/catalog/google-weathernext3-forecast-15-day-0-1-degree-virtual/

Real-time data and forecasts are subject to Google's [GDM Real-Time Weather Forecasting Experimental Data Terms of Use](https://storage.googleapis.com/weathernext-public/terms-of-use.pdf); historical data are licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Read the [terms notice](https://google-weathernext3.r2.dynamical.org/google-weathernext3-forecast-15-day-0-1-degree-virtual/TERMS_OF_USE.pdf) and [binding notice](https://google-weathernext3.r2.dynamical.org/google-weathernext3-forecast-15-day-0-1-degree-virtual/LEGALLY_BINDING_TERMS_OF_USE.txt), and retain the dataset's attribution and [third-party attributions and acknowledgements](https://storage.googleapis.com/weathernext-public/weathernext-3-attributions-acknowledgements.pdf), whose terms may also apply.

This notebook uses Icechunk through `dynamical_catalog.open`.


In [ ]:
# If running locally, follow README.md for dependency installation.
# If using Google Colab, run this cell.
!uv pip install 'dynamical-catalog>=1.0.0' 'icechunk>=2.1.2' matplotlib cartopy 'zarr>=3.0'

In [ ]:
import cartopy.crs as ccrs
import dynamical_catalog
import zarr

# Keep concurrent chunk reads bounded.
zarr.config.set({"async.concurrency": 1})
ds = dynamical_catalog.open(
    "google-weathernext3-forecast-15-day-0-1-degree-virtual", chunks=None
)
ds

## Select the mean on the native grid

The horizontal dimensions are `y` (latitude, ascending south to north) and `x` (longitude, 0–360°). For example, 105°W is `x=255`; regional slices below stay within that native coordinate convention. Select `statistic="mean"` explicitly rather than averaging across mean, p10, p25, p50, p75 and p90.

Temperature reads in degrees Celsius; no Kelvin conversion is needed. Recent initializations may contain NaNs while source files arrive. The example uses a fixed archived initialization and one lead, and loads one regional temperature field for reuse below.


In [ ]:
print(ds["temperature_2m"].attrs["units"])
print(ds.attrs["license"])
print(ds.attrs["attribution"])

temperature = (
    ds["temperature_2m"]
    .sel(
        init_time="2026-10-21T00",
        lead_time="24h",
        statistic="mean",
        y=slice(25, 55),
        x=slice(230, 300),
    )
    .load()
)

## A day-ahead temperature field over North America

This is the forecast initialized October 21, 2026 at 00:00 UTC, valid October 22 at 00:00 UTC. It shows a model forecast, not an observation or a measure of forecast skill.


In [ ]:
temperature_map = temperature.plot.pcolormesh(
    x="x",
    y="y",
    figsize=(10, 5),
    cmap="coolwarm",
    center=False,
    subplot_kws={"projection": ccrs.PlateCarree()},
    transform=ccrs.PlateCarree(),
)
temperature_map.axes.coastlines()

## Inspect the nearest grid cell to Denver

The point is selected from the field already loaded above. The returned coordinates identify the resolved grid cell, which need not coincide with a weather station; terrain and grid resolution matter when comparing it to observations.


In [ ]:
denver = temperature.sel(y=39.74, x=(-104.99) % 360, method="nearest")
denver

## Interpretation and next steps

Use the catalog's validation report for the tested scope and known limitations. These are forecasts, not official warnings; consult the relevant national meteorological service for safety decisions.

Consult the validation report’s review notes before relying on sea-surface temperature.

- How does the same initialized forecast compare with independent observations at the same valid time and resolved grid cell?
- How do the precomputed percentile fields differ from the mean? Select a single label for each comparison; their spread is not an independently calibrated probability estimate.
